<a href="https://colab.research.google.com/github/jvictorferreira3301/metodos-numericos/blob/main/2_matrizes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho 2 - Sistemas de Equações Lineares

## 1. Entrada de Dados do Sistema

*Objetivo:* O programa receberá a entrada do usuário contendo as informações relevantes do sistema, seja na forma matricial ou de equações. O código deve aceitar matrizes de qualquer dimensão ($m \times n$), sem impor restrições de tamanho.

In [23]:
import numpy as np

def entrada_por_equacoes():
    """
    Usuário digita cada equação no formato:
    2*x1 + 3*x2 - x3 = 5
    O programa extrai os coeficientes automaticamente.
    """
    from sympy import symbols, Eq, sympify, Symbol
    from sympy.parsing.sympy_parser import parse_expr

    n_equacoes = int(input("Quantas equações o sistema possui? "))
    n_variaveis = int(input("Quantas variáveis o sistema possui? "))

    # Cria variáveis x1, x2, x3, ..., xn
    variaveis = symbols(f'x1:{n_variaveis + 1}')
    print(f"\nVariáveis disponíveis: {list(variaveis)}")
    print("Exemplo de equação: 2*x1 + 3*x2 - x3 + 2")

    A = []  # Matriz de coeficientes
    b = []  # Vetor de termos independentes

    for i in range(n_equacoes):
        print(f"\nEquação {i+1}")
        lado_esq = input("  Lado esquerdo (ex: 2*x1 + 3*x2 - x3 + 2): ")
        lado_dir = float(input("  Lado direito  (ex: 5): "))

        # Analisa o lado esquerdo
        expr_esq = parse_expr(lado_esq, local_dict={str(v): v for v in variaveis})

        # Cria uma expressão única: Lado_Esquerdo - Lado_Direito = 0
        expr_total = expr_esq - lado_dir
        
        # Exibe a equação escolhida para confirmação
        print(f"Lida com sucesso: {expr_esq} = {lado_dir}")

        # Extrai os coeficientes de cada variável para a matriz A
        linha = []
        for var in variaveis:
            coef = expr_total.coeff(var)
            linha.append(float(coef))

        # Encontra a constante solta substituindo todas as variáveis por zero
        constante = expr_total.subs({v: 0 for v in variaveis})

        # Como a expressão é (Variáveis) + Constante = 0,
        # passamos a constante para o lado direito invertendo o sinal
        A.append(linha)
        b.append(float(-constante))

    A_np = np.array(A)
    b_np = np.array(b).reshape(-1, 1)

    # Matriz aumentada [A | b]
    aumentada = np.hstack([A_np, b_np])

    return A_np, b_np, aumentada

def entrada_por_matriz():

    """
    Usuário digita diretamente os elementos da matriz aumentada [A | b].
    """
    m = int(input("Digite o número de linhas (equações): "))
    n = int(input("Digite o número de colunas (variáveis + termo independente): "))

    print(f"\nDigite a matriz aumentada [{m}x{n}]")
    print("(cada linha: coeficientes + termo independente separados por espaço)\n")

    matriz = []
    for i in range(m):
        linha = [float(x) for x in input(f"Linha {i+1}: ").split()]

        # Valida se o número de elementos está correto
        while len(linha) != n:
            print(f"Esperado {n} valores, recebido {len(linha)}. Tente novamente.")
            linha = [float(x) for x in input(f"Linha {i+1}: ").split()]

        matriz.append(linha)

    aumentada = np.array(matriz)
    A_np = aumentada[:, :-1]
    b_np = aumentada[:, -1].reshape(-1, 1)

    return A_np, b_np, aumentada


def exibir(A, b, aumentada):
    print("\n" + "="*50)
    print("SISTEMA RECEBIDO")
    print("="*50)
    print(f"\nMatriz de coeficientes A [{A.shape[0]}x{A.shape[1]}]:")
    print(A)
    print(f"\nVetor de termos independentes b [{b.shape[0]}x1]:")
    print(b)
    print(f"\nMatriz aumentada [A|b] [{aumentada.shape[0]}x{aumentada.shape[1]}]:")
    print(aumentada)


# ══════════════════════════════════════════════════════
# EXECUÇÃO PRINCIPAL
# ══════════════════════════════════════════════════════

print("="*50)
print("ENTRADA DO SISTEMA LINEAR")
print("="*50)
print("\nComo deseja fornecer o sistema?")
print("1. Por equações")
print("2. Por matriz")

opcao = input("\nEscolha (1 ou 2): ").strip()

if opcao == '1':
    A, b, aumentada = entrada_por_equacoes()
elif opcao == '2':
    A, b, aumentada = entrada_por_matriz()
else:
    print("Opção inválida.")
    exit()

exibir(A, b, aumentada)

ENTRADA DO SISTEMA LINEAR

Como deseja fornecer o sistema?
1. Por equações
2. Por matriz



Variáveis disponíveis: [x1, x2]
Exemplo de equação: 2*x1 + 3*x2 - x3 + 2

Equação 1
Lida com sucesso: x1 + 2*x2 = 2.0

Equação 2
Lida com sucesso: 8*x1 - x2 = 10.0

SISTEMA RECEBIDO

Matriz de coeficientes A [2x2]:
[[ 1.  2.]
 [ 8. -1.]]

Vetor de termos independentes b [2x1]:
[[ 2.]
 [10.]]

Matriz aumentada [A|b] [2x3]:
[[ 1.  2.  2.]
 [ 8. -1. 10.]]


## 2. Eliminação de Gauss: Forma Escalonada

*Objetivo:* Realizar uma sequência de Operações Elementares com Linhas de forma totalmente manual (sem uso de funções de bibliotecas existentes) para obter a matriz escalonada. O código deverá resolver o caso geral do sistema (sem solução, solução única ou infinitas soluções). Para isso:

#### 2.1 Permutação de Linhas (Pivotamento)

*Objetivo:* Antes de operar, checar se o pivô é zero. Caso seja, realizar a permuta pela primeira linha abaixo que possua valor não nulo na mesma coluna. Se todos abaixo também forem nulos, avançar para a próxima coluna sem alterações.

#### 2.2 Otimização da Eliminação

*Objetivo:* Garantir que a anulação dos elementos abaixo do pivô seja otimizada. O programa não deve fazer operações desnecessárias se o elemento já for nulo, pulando direto para a próxima linha.

#### 2.3 Tratamento de Linhas Inconsistentes

*Objetivo:* Identificar a ocorrência de equações inconsistentes durante o processo. O programa deve parar a execução imediatamente e avisar ao usuário que o sistema não possui solução.

#### 2.4 Remoção de Linhas Nulas

*Objetivo:* Caso o sistema não apresente inconsistências, mas possua linhas completamente nulas, o programa deverá remover essas linhas da matriz.

#### 2.5 Impressão Parcial e Final da Matriz

*Objetivo:* Imprimir a matriz atualizada para o usuário sempre que todas as operações de uma coluna forem finalizadas, antes de passar para a próxima. Ao fim do processo, o usuário deve ser informado de que a forma escalonada foi obtida e ela deve ser exibida.




In [26]:
def escalonar_matriz(matriz_aumentada):
    """
    Realiza a Eliminação de Gauss respeitando todos 
    os requisitos.
    """
    matriz = matriz_aumentada.astype(float).copy()
    linhas = matriz.shape[0]
    colunas = matriz.shape[1]
    linha_atual = 0  # Controla a linha onde o pivô deveria estar
    
    tol = 1e-10  # Tolerância para evitar erros de ponto flutuante (arredondamento)

    print("\n" + "═"*50)
    print("INICIANDO ESCALONAMENTO (ELIMINAÇÃO DE GAUSS)")
    print("═"*50)

    # O laço principal varre as colunas, exceto a última (que é o vetor 'b')
    for j in range(colunas - 1):
        
        # Se já passamos de todas as linhas possíveis para pivô, paramos
        if linha_atual >= linhas:
            break
            
        # ------------------------------------------------------------------
        # REQUISITO 2.1: Permuta de linha se o pivô for nulo
        # ------------------------------------------------------------------
        if abs(matriz[linha_atual][j]) < tol:
            linha_troca = -1
            # Procura a primeira linha abaixo com elemento não nulo na mesma coluna
            for k in range(linha_atual + 1, linhas):
                if abs(matriz[k][j]) > tol:
                    linha_troca = k
                    break
            
            if linha_troca != -1:
                print(f"\nTrocando linha {linha_atual+1} com a linha {linha_troca+1} (Pivô zero encontrado na coluna {j+1})")

                for c in range(colunas):
                    temp = matriz[linha_atual][c]
                    matriz[linha_atual][c] = matriz[linha_troca][c]
                    matriz[linha_troca][c] = temp
            else:
                # Todos os elementos abaixo são nulos. Pula para a próxima coluna sem fazer nada.
                continue
        
        # ------------------------------------------------------------------
        # REQUISITO 2.2: Otimização (Pular elementos já nulos abaixo do pivô)
        # ------------------------------------------------------------------
        operacao_na_coluna = False
        for i in range(linha_atual + 1, linhas):
            if abs(matriz[i][j]) < tol:
                continue  # Pula para a próxima linha: o elemento já é nulo!
            
            # Calcula o fator multiplicador
            fator = matriz[i][j] / matriz[linha_atual][j]
            
            for c in range(j, colunas):
                matriz[i][c] = matriz[i][c] - (fator * matriz[linha_atual][c])
            
            operacao_na_coluna = True
        
        # ------------------------------------------------------------------
        # REQUISITO 2.5: Imprimir a matriz após operações na coluna
        # ------------------------------------------------------------------
        if operacao_na_coluna:
            print(f"\nMatriz após operações na coluna {j+1}:")
            # Arredondando para 4 casas decimais
            print(np.round(matriz, 4)) 
        
        # Avança o pivô para a próxima linha
        linha_atual += 1

    # ------------------------------------------------------------------
    # REQUISITOS 2.3 e 2.4: Varredura de Limpeza Final
    # ------------------------------------------------------------------
    linhas_validas = []
    
    for i in range(linhas):
        # Verifica se todos os coeficientes (tudo exceto a última coluna) são zero
        coeficientes_zerados = all(abs(matriz[i][c]) < tol for c in range(colunas - 1))
        termo_independente = matriz[i][-1]
        
        if coeficientes_zerados:
            # REQUISITO 2.3: Linha inconsistente (ex: 0x1 + 0x2 = 5)
            if abs(termo_independente) > tol:
                print("\nPARADA OBRIGATÓRIA: O sistema não possui solução!")
                print(f"Linha inconsistente detectada: 0 = {termo_independente:.4f}")
                return None  # Interrompe tudo e retorna vazio
            
            # REQUISITO 2.4: Remover linhas totalmente nulas
            else:
                print(f"\nRemovendo a linha {i+1} pois ela se tornou totalmente nula (0 = 0).")
        
        else:
            linhas_validas.append(i)
            
    # Remontando a matriz apenas com as linhas válidas
    matriz_final = []
    for indice in linhas_validas:
        matriz_final.append(matriz[indice])
    
    matriz_escalonada = np.array(matriz_final)

    # ------------------------------------------------------------------
    # REQUISITO 2.6: Informar usuário e imprimir a matriz escalonada
    # ------------------------------------------------------------------
    print("\n" + "═"*50)
    print("MATRIZ ESCALONADA OBTIDA COM SUCESSO!")
    print("═"*50)
    print(np.round(matriz_escalonada, 4))
    
    return matriz_escalonada

# =======================================================
# EXECUÇÃO PRINCIPAL
# =======================================================

# Verifica se a matriz 'aumentada' foi criada na Célula 1
if 'aumentada' in locals() or 'aumentada' in globals():
    # Chama a função passando a matriz e guarda o resultado
    matriz_resultante = escalonar_matriz(aumentada)
else:
    print("A matriz 'aumentada' não foi encontrada!")
    print("Você precisa rodar o código de leitura dos dados primeiro.")


══════════════════════════════════════════════════
INICIANDO ESCALONAMENTO (ELIMINAÇÃO DE GAUSS)
══════════════════════════════════════════════════

Matriz após operações na coluna 1:
[[  1.   2.   2.]
 [  0. -17.  -6.]]

══════════════════════════════════════════════════
MATRIZ ESCALONADA OBTIDA COM SUCESSO!
══════════════════════════════════════════════════
[[  1.   2.   2.]
 [  0. -17.  -6.]]


## 3. Eliminação de Gauss-Jordan: Forma Canônica

*Objetivo:* A partir da forma escalonada resultante da etapa anterior, realizar operações elementares regressivas até alcançar e imprimir a forma canônica da matriz.

In [ ]:
# Implementação para a eliminação de Gauss-Jordan (forma canônica)

## 4. Análise e Retorno da Solução do Sistema

*Objetivo:* A partir da matriz na forma canônica, informar qual é o tipo de solução do sistema (única ou infinitas).

### 4.1 Sistema com Solução Única

*Objetivo:* Se o sistema apresentar solução única, os valores finais das variáveis devem ser retornados ao usuário.

In [ ]:
# Implementação para o sistema com solução única

### 4.2 Sistema com Infinitas Soluções

*Objetivo:* Se o sistema apresentar infinitas soluções, a rotina deverá calcular a quantidade de variáveis livres e retornar ao usuário a forma geral da solução parametrizada.

In [ ]:
# Implementação para o sistema com infinitas soluções

## 5. Aplicações da Rotina de Eliminação (Exemplos do PDF)

*Objetivo:* Aplicar o programa principal desenvolvido nas seções anteriores para mostrar todas as possibilidades de resolução propostas no material em PDF.

### 5.1 Exemplo: Sistema Inconsistente

*Objetivo:* Utilizar o programa para resolver o problema 2.2 (pág. 29 do PDF), demonstrando um caso sem solução.

In [ ]:
# Implementação para o exemplo de sistema inconsistente

### 5.2 Exemplo: Solução Única

*Objetivo:* Utilizar o programa para resolver o problema 2.3 (pág. 29 do PDF), demonstrando um caso de solução única.

In [ ]:
# Implementação para o exemplo de solução única

### 5.3 Exemplo: Infinitas Soluções (Uma variável livre)

*Objetivo:* Utilizar o programa para resolver o problema 2.3 (págs. 29 e 30 do PDF), apresentando o caso com uma variável livre.

In [ ]:
# Implementação para o exemplo de infinitas soluções (uma variável livre)

### 5.4 Exemplo: Infinitas Soluções (Duas variáveis livres)

*Objetivo:* Utilizar o programa para resolver o problema 2.1 (pág. 28 do PDF), apresentando o caso de múltiplas variáveis livres.

In [ ]:
# Implementação para o exemplo de infinitas soluções (duas variáveis livres)

## 6. Fatoração e Decomposição LU

*Objetivo:* Implementar uma rotina específica para executar a Decomposição LU em matrizes quadradas de ordem qualquer.

### 6.1 Passo a Passo e Comparação com Gauss

*Objetivo:* O algoritmo deve mostrar todos os passos da decomposição. Além disso, deve resolver sistemas de solução única e ter seus resultados comparados com a rotina principal de eliminação de Gauss.

In [ ]:
# Implementação para o passo a passo e comparação da decomposição LU com Gauss

### 6.2 Matrizes Inversas e Problemas Específicos

*Objetivo:* Empregar a Decomposição LU para encontrar matrizes inversas, com o teste obrigatório de que $[A][A]^{-1} = I$ usando o exemplo 10.3 (pág. 237). O programa também deve resolver o problema 10.8 da pág. 244.

In [ ]:
# Implementação para matrizes inversas e problemas específicos com decomposição LU

### 6.3 Vantagens da Decomposição LU

*Objetivo:* Demonstrar e exemplificar matematicamente as vantagens no uso computacional da decomposição LU em relação a outros métodos.

In [ ]:
# Implementação para demonstrar as vantagens da decomposição LU

## 7. Métodos Iterativos: Gauss-Seidel e Jacobi

*Objetivo:* Estruturar algoritmos iterativos para encontrar soluções aproximadas de sistemas lineares.

### 7.1 Implementação de Gauss-Seidel e Jacobi

*Objetivo:* Escrever o método padrão de Gauss-Seidel, bem como o método através da iteração de Jacobi.

In [ ]:
# Implementação dos métodos de Gauss-Seidel e Jacobi

### 7.2 Testes e Validação dos Algoritmos

*Objetivo:* Resolver os mesmos sistemas exemplificados na etapa da Decomposição LU, agora utilizando os métodos iterativos criados.

In [ ]:
# Implementação para testes e validação dos algoritmos iterativos

## 8. Sistemas com Matrizes Tridiagonais

*Objetivo:* Criar algoritmos voltados especificamente para fazer a decomposição e solucionar sistemas cuja matriz de coeficientes possua estrutura de banda tridiagonal.

### 8.1 Implementação do Algoritmo de Thomas

*Objetivo:* Codificar o Algoritmo de Thomas e utilizá-lo para resolver o exemplo 11.1 (pág. 249 do livro base).

In [ ]:
# Implementação do Algoritmo de Thomas

## 9. Sistemas com Matrizes Simétricas

*Objetivo:* Criar funções voltadas à decomposição e resolução de sistemas que apresentem matrizes simétricas.

### 9.1 Decomposição de Cholesky

*Objetivo:* Codificar o algoritmo da fatoração de Cholesky e testá-lo resolvendo o exemplo 11.2 (pág. 249 do livro base).

In [ ]:
# Implementação da decomposição de Cholesky

## 10. Análise de Desempenho e Eficiência

*Objetivo:* Monitorar o custo computacional avaliando a execução geral das rotinas.

### 10.1 Registro de Tempo e Operações

*Objetivo:* Criar um mecanismo capaz de contabilizar o número de operações aritméticas realizadas pelos algoritmos ou cronometrar o tempo total de execução.

In [ ]:
# Implementação para o registro de tempo e operações

### 10.2 Estudo Comparativo de Métodos

*Objetivo:* Submeter diversos exemplos ao algoritmo mais adequado para aquele formato específico de matriz, resolvendo logo em seguida o mesmo problema utilizando o modelo de Eliminação de Gauss padrão. Avaliar e comparar a eficiência obtida pelos métodos envolvidos.

In [ ]:
# Implementação para o estudo comparativo de métodos